# Lab 2 – Predicting Diabetes Progression

**Objective:** build a model that best predicts *disease progression one year after baseline*,
to be used as a screening tool that helps physicians identify patients at risk.

**Models:** univariate polynomial regression (BMI), multivariate polynomial regression,
decision trees, k-Nearest Neighbours and logistic regression.

**Metrics:** R², Mean Absolute Error (MAE), Mean Absolute Percentage Error (MAPE) on a
75 % / 10 % / 15 % train / validation / test split.

# Part 1 – Data
## 1. Get the data

`load_diabetes` options used:
- `as_frame=True` – returns a pandas DataFrame with named columns.
- `scaled=False` – keeps the **original units** (e.g. BMI in kg/m², age in years) so the
  BMI equation and predictions in Part 2 are clinically interpretable. The default
  (`scaled=True`) mean-centres each feature and divides by `std * sqrt(n_samples)`.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn import datasets

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 3)

In [2]:
diabetes = datasets.load_diabetes(as_frame=True, scaled=False)
print(diabetes.DESCR[:1500])

.. _diabetes_dataset:

Diabetes dataset
----------------

Ten baseline variables, age, sex, body mass index, average blood
pressure, and six blood serum measurements were obtained for each of n =
442 diabetes patients, as well as the response of interest, a
quantitative measure of disease progression one year after baseline.

**Data Set Characteristics:**

:Number of Instances: 442

:Number of Attributes: First 10 columns are numeric predictive values

:Target: Column 11 is a quantitative measure of disease progression one year after baseline

:Attribute Information:
    - age     age in years
    - sex
    - bmi     body mass index
    - bp      average blood pressure
    - s1      tc, total serum cholesterol
    - s2      ldl, low-density lipoproteins
    - s3      hdl, high-density lipoproteins
    - s4      tch, total cholesterol / HDL
    - s5      ltg, possibly log of serum triglycerides level
    - s6      glu, blood sugar level

Note: Each of these 10 feature variables have bee

The six blood-serum columns are named `s1`–`s6`. The dataset description tells us what they
measure, so we rename them to make the rest of the report easier to read.

In [3]:
column_names = {
    "s1": "tc",   # total serum cholesterol
    "s2": "ldl",  # low-density lipoproteins
    "s3": "hdl",  # high-density lipoproteins
    "s4": "tch",  # total cholesterol / HDL
    "s5": "ltg",  # log of serum triglycerides level
    "s6": "glu",  # blood sugar level
}
df = diabetes.frame.rename(columns=column_names)
df.head()

,age,sex,bmi,bp,tc,ldl,hdl,tch,ltg,glu,target
0,59.0,2.0,32.1,101.0,157.0,93.2,38.0,4.0,4.860,87.0,151.0
1,48.0,1.0,21.6,87.0,183.0,103.2,70.0,3.0,3.892,69.0,75.0
2,72.0,2.0,30.5,93.0,156.0,93.6,41.0,4.0,4.673,85.0,141.0
3,24.0,1.0,25.3,84.0,198.0,131.4,40.0,5.0,4.890,89.0,206.0
4,50.0,1.0,23.0,101.0,192.0,125.4,52.0,4.0,4.290,80.0,135.0


In [4]:
df.shape

(442, 11)

## 2. Frame the problem

**Target:** `target` – a quantitative measure of *disease progression one year after baseline*
(higher = worse progression). **Inputs:** 10 baseline measurements (age, sex, BMI, average
blood pressure and six blood-serum measurements) for 442 patients.

**Task type:** supervised learning, offline/batch (small, static dataset).
- As stated, it is a **regression** problem – we predict a continuous score
  (polynomial regression, regression trees, kNN regression).
- Because the end use is *screening* (flag patients at risk), it can also be framed as
  **binary classification**: "high progression" vs "low progression". This is how logistic
  regression is used in Part 3.

**Talking points from the workshops**
- *Performance metrics (classification):* in screening, a **false negative** (an at-risk
  patient who is not flagged) is more costly than a false positive (an extra follow-up test),
  so **recall / sensitivity** matters more than raw accuracy. A confusion matrix shows these
  error types separately, and the decision threshold can be lowered to trade precision for recall.
- *k-Nearest Neighbours:* predictions are the average (or vote) of the *k* most similar
  patients. It is distance-based, so features must be **scaled**, and it degrades as the number
  of features grows (curse of dimensionality). Small *k* → low bias / high variance; large *k*
  → smoother, higher bias.
- *Logistic regression:* models the **probability** of the positive class through the sigmoid
  of a linear combination of features. Probabilities are useful for screening because the
  physician can choose the risk threshold, and coefficients are interpretable (log-odds).

**Regression metrics used here**
- **R²** – share of the target's variance explained by the model (1 = perfect, 0 = no better
  than predicting the mean, can be negative).
- **MAE** – average absolute error, in the same units as the target.
- **MAPE** – average absolute error relative to the true value (scale-free, %); sensitive to
  small true values.

**Baseline:** a model that always predicts the training mean (this is exactly the degree-0
polynomial in Part 2). Any useful model must beat it.

> 📝 *Explanation to expand:* clinical context / why progression prediction matters.